# Survivor EvalCache Demo

This notebook demonstrates how the Survivor test stage can leverage an EvaluationTool cache despite its unique method of breaking down Datasets into single objects for calculating per-datum metrics.

First, we set up a model, (small) dataset, metric, and evaluation tool *with cache* for Object Detection.

In [ ]:
### Create model wrapper
from jatic_ri.object_detection.models import TorchvisionODModel
model_name = "ssdlite320_mobilenet_v3_large"
model_wrapper = TorchvisionODModel(model_name=model_name)

### Create dataset wrapper
import os
from pathlib import Path
from jatic_ri import PACKAGE_DIR
from jatic_ri.object_detection.datasets import CocoDetectionDataset
coco_dataset_dir = PACKAGE_DIR.parent.parent.joinpath(Path('tests/testing_utilities/example_data/coco_resized_val2017'))
dataset_wrapper = CocoDetectionDataset(
    root=str(coco_dataset_dir),
    ann_file=str(coco_dataset_dir.joinpath('instances_val2017_resized_6.json')),
)

### Create metric
from jatic_ri.object_detection.metrics import map50_torch_metric_factory
metric_wrapper = map50_torch_metric_factory()

### Create evaluation tool
from jatic_ri.util.evaluation import EvaluationTool
from jatic_ri.util.cache import SimpleRICacheOD

tmp_dir = os.path.join(os.getcwd(), 'cache_dir')
eval_tool = EvaluationTool(SimpleRICacheOD(cache_root_dir=tmp_dir, compress_json=False))

Next, set up the Survivor OD test stage.

In [ ]:
### Generate Test Stage configurations 
# generate xaitk config based on defaults from its config app
from jatic_ri.object_detection._panel.configurations.survivor_app import SurvivorApp as SurvivorAppOD
survivor_app = SurvivorAppOD()
survivor_app._run_export()
survivor_config = survivor_app.output_test_stages["survivor_test_stage"]

# construct the config
full_dashboard_config = {
    'task': 'object_detection',
    'survivor_config': survivor_config, 
}

# Even though the model object is the same, EvalTool will use the dict keys as model_id so these will store separately in cache.
# NOTE: This won't work once upgraded to MAITE 0.7.0 since the model ID will be part of the Model object

models = {
    "model1": model_wrapper,
    "model2": model_wrapper
}

### Instantiate Survivor Test Stage and load components
from jatic_ri.util.dashboard_utils import rehydrate_test_stage_od
test_stage = rehydrate_test_stage_od(config=survivor_config)
test_stage.load_models(models)
test_stage.load_dataset(dataset_wrapper, "dataset1")
test_stage.load_metric(metric_wrapper, metric_wrapper.return_key)
test_stage.load_eval_tool(eval_tool)

Next, we use the EvaluationTool to run `predict` on the first model with dataset.  This recreates the state the cache would be in if a previous test stage had already run predictions with `model1` on `dataset1`.

After this step is complete, there will be a `model1_dataset1_1.json` file in the cache directory.

In [ ]:
_,_ = eval_tool.predict(model_id="model1",model=model_wrapper,dataset_id="dataset1",dataset=dataset_wrapper)
os.listdir(tmp_dir)

Next, we run the test stage.

Note `use_stage_cache=False` argument tells the test stage to skip the stage-specific SurvivorTestStageCache, but the EvaluationTool (common to all test stages in a workflow) will still be used.

As Survivor analyzes the images for model1, it reads `model1_dataset1_1.json` from cache.  It still computes metrics on a per-image basis and saves cache enties in the format `model1_dataset1-img[X]_map_50_1.json`, but avoids having to run the model to generate predictions.

Since model2+dataset1 predictions are not in cache, the 'predict' step will have to run the model and  `model1_dataset2_1.json` will be generated along with the individual image/metric files.

In [ ]:
_ = test_stage.run(use_stage_cache=False)
os.listdir(tmp_dir)